# Exp 030 — CoT user-state prompt + Qwen 7B (BLIND-A)

**Pair-test sibling of 029 Blind-A.** Same retrieval (wRRF), same CoT prompt, same `max_new_tokens=128`. Only `lm_type` changes: Qwen 2.5-1.5B-Instruct → **Qwen 2.5-7B-Instruct**.

**Hypothesis**: 7B follows the structured CoT format more reliably (~95% expected vs 1.5B's measured 67%), so the user_state extraction is more consistently producing real signal that the response can ground in. Combined with the prompt's word-bans on filler ('fantastic', 'perfectly'), the prior 7B failure mode (exp prior-branch -0.45 LLM judge) should be neutralised.

**Risk**: if 7B+CoT *still* regresses on Gemini despite the word-bans, the bigger-model penalty is structural — 1.5B stays as production, and we revert to 029 as the candidate.

**Inference path**: vanilla HF generate (NOT vLLM). 80 rows is too small for vLLM's throughput advantage to repay its setup cost + the dependency/CUDA pain we hit before. 7B + batch 8 + sdpa fits A100 40GB comfortably (~22 GB total: 14 GB weights + 4 GB KV + 4 GB activations).

Wall time on A100: ~3-5 min for 80 rows.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 2b) Mount Drive + wire persistent caches.
# Persists across Colab sessions (clear wins for the Blind-A 80-row run):
#   * HF datasets (talkpl-ai/* — track metadata, user profiles,
#     Blind-A split). Otherwise re-downloads ~30-60 sec each session.
#   * experiments/cache (BM25 + dense + cf-bpr indices). Otherwise
#     rebuilds 2-3 min each session.
# Does NOT persist:
#   * HF model weights (Drive read slower than HF download for ~3-14 GB)
#
# Drive auth: a popup appears the first time. CLICK THROUGH ALL
# permission screens. We retry with force_remount=True if the first
# attempt fails (common cause: 'credential propagation was unsuccessful'
# from closing the OAuth popup early).
import os, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)

assert os.path.isdir('/content/drive/MyDrive'), (
    'Drive mount failed — /content/drive/MyDrive does not exist. '
    'Common fixes: complete the OAuth popup fully, disable popup '
    'blocker, or sign in to Google in this browser tab on the same account.'
)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026-lora-tutorial/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')
!ls -lh {DRIVE_BASE}/

In [ ]:
# 3) Install deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters.
TID = '030-cot-user-state-qwen7b-blindsetA'
# 7B at bf16+sdpa: batch 8 fits A100 40GB with comfortable headroom.
# Drop to 4 if you somehow OOM (shouldn't happen for 80 rows).
BATCH_SIZE = 8
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run Blind-A inference.
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package prediction.zip per CodaBench spec.
# CodaBench requires: zip with `prediction.json` (singular) at archive root.
# Server reads /app/input/res/prediction.json. Any other layout = rejected.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 80 for Blind-A)')
assert len(rows) == 80, f'wrong row count {len(rows)} — do not ship'

# Schema check on first row.
sample = rows[0]
required = {'session_id', 'user_id', 'turn_number', 'predicted_track_ids', 'predicted_response'}
missing = required - set(sample.keys())
assert not missing, f'missing fields: {missing}'
assert len(sample['predicted_track_ids']) == 20, f'expected 20 track_ids, got {len(sample["predicted_track_ids"])}'
assert sample['predicted_response'].strip(), 'first row has empty response'
print(f'sample response[0]: {sample["predicted_response"][:300]!r}')

# Stage as singular prediction.json then zip.
stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip

In [ ]:
# 7a) Browser download — this is the file you upload to CodaBench.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Drive backup (also saves the raw 80-row JSON for local scoring).
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst} | head -20

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate on the 80 rows.
# Same checks as the dev-set notebook. The parser should produce clean
# response prose; if you see <user_state> tags or 'mood:' field names
# leaking through, that's a parser bug — DO NOT submit.
import json, random, re

with open(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1; continue
    if field_leak_re.search(resp): leak_field += 1
    if tag_leak_re.search(resp): leak_tag += 1

n = len(rows)
print(f'rows total      : {n}')
print(f'empty responses : {empty}  ({empty/n:.1%})')
print(f'field-name leak : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak        : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== ALL 80 sample responses (read these before uploading) ===')
for i in range(min(15, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

## Submit to CodaBench

1. Cell 7a downloads `prediction.zip`.
2. CodaBench → Submit a new entry → upload `prediction.zip`.
3. Wait ~1-3 min for scoring.
4. Update `documents/submissions_log.md` and `documents/experiments_log.md`.

## Decision gate after CodaBench scores (compared to 029 Blind-A and exp 024 / 3B-stock-prompt)

- **030 composite > 029 composite by ≥0.03** → 7B+CoT compounds the CoT effect; 7B+CoT is the new candidate stack.
- **030 composite within ±0.03 of 029** → noise; 7B doesn't add lift over 1.5B with CoT. 1.5B stays cheaper for production. CoT itself is the dominant lever.
- **030 composite < 029 composite by ≥0.03** → bigger-model penalty is structural, even with CoT word-bans. Mark 7B+CoT as falsified in `project_fresh_model_state.md`.